In [ ]:
from langchain.agents import create_agent
from langchain_ollama import ChatOllama
from langchain.messages import HumanMessage
from pprint import pprint

In [ ]:
llm = ChatOllama(model="gemma4:31b-cloud")
# agent = create_agent(model=llm)

In [ ]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content=f"Read this resume and extract key fields and return a structured json. Resume:{text}")]}
)

# pprint(response)
print(response['messages'][-1].content)

In [2]:
import fitz  # This imports PyMuPDF
doc = fitz.open("Resume.pdf")
text = doc[0].get_text()

In [ ]:
text

In [ ]:
response = llm.invoke(f"Read this resume and extract key fields and return a structured json. Resume:{text}")
print(response)

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [7]:
from pydantic import BaseModel
from langchain.agents import create_agent
from langchain_ollama import ChatOllama
from langchain.messages import HumanMessage
from pprint import pprint
from typing import List, Dict
import json
from google import genai
from pydantic import BaseModel, Field
from typing import List, Optional

i:\Projects\AIJent\AIJent\.venv\Lib\site-packages\langgraph\checkpoint\serde\encrypted.py:5: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


In [16]:
class Experience(BaseModel):
    Title: str
    Company: str
    Location: str
    Start_Date: str
    End_Date: str
    Description: str

class Education(BaseModel):
    Degree: str
    Major: str
    Instituion: str
    Location: str
    Start_Date: str
    End_Date: str
    
class Projects(BaseModel):
    Title: str
    Tools: str
    Description: str

class Profile(BaseModel):
    first_name: str
    last_name: str
    phone_number: str
    email: str
    linkedin: str
    github: str
    summary: str
    skills: List[str]
    years_of_exp: int
    experience: List[Experience]
    projects: List[Projects]
    education: List[Education]


In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

# model = ChatGoogleGenerativeAI(model="gemma-4-31b-it")
model = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite")
structured_model = model.with_structured_output(
    schema=Profile.model_json_schema(), method="json_schema"
)

# response = structured_model.invoke("The new UI is great!")
# agent = create_agent(model=model,
#                      system_prompt="You are a resume parser agent that takes in a resume and extracts info from it. Do not summarize any info from the resume. Extract the text exactly as it is and return it in json format.",
#                      response_format=Profile)

# response = agent.invoke(
#     {"messages": [HumanMessage(content=f"extract all info from this resume: {text}")]}
# )
# print(response['messages'][-1].content)

In [22]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a resume parser agent that takes in a resume and extracts info from it. Do not summarize any info from the resume. Extract the text exactly as it is and return structured feedback."),
    ("human", "{input}")
])

chain = prompt | structured_model

In [ ]:
result = chain.invoke({"input": text})
print(result)

In [20]:

with open("output.json", "w") as f:
    json.dump(result, f, indent=2)

In [ ]:
client = genai.Client()
response = client.models.generate_content(
    model="gemma-4-31b-it",
    contents=f"Parse this resume: {text}",
    config={
        "response_format": {"text": {"mime_type": "application/json", "schema": Profile.model_json_schema()}},
    },
)

In [8]:
llm = ChatOllama(model="gemma4:31b-cloud", format="json")
structured_llm = llm.with_structured_output(Profile)
# agent = create_agent(
#     model=llm,
#     system_prompt="You are a resume parser agent that takes resume text and extracts all information and outputs it to the proper fields",
#     response_format= Profile
# )

In [ ]:
response = structured_llm.invoke(f"Parse this resume: {text}")

In [ ]:
response